# Введение в декораторы. Фундаментальные концепции

## Что такое декораторы?

### Шаг 1: Общее определение. Концепция "обёртки"

Представьте, что у вас есть несколько функций в проекте. Например, одна скачивает данные из интернета, другая обрабатывает файл, третья выполняет сложные вычисления. И вот появляется задача: для каждой из этих функций нужно замерять время выполнения.

Первое, что приходит в голову - просто добавить код для замера времени в начало и конец каждой функции.

In [1]:
import time

def download_data():
    start_time = time.time() # Начало замера
    print("Начинаю скачивание...")
    # ... какая-то долгая операция ...
    print("Скачивание завершено.")
    end_time = time.time() # Конец замера
    print(f"Функция выполнялась {end_time - start_time} секунд.")

def process_file():
    start_time = time.time() # Начало замера
    print("Начинаю обработку файла...")
    # ... какая-то долгая операция ...
    print("Обработка завершена.")
    end_time = time.time() # Конец замера
    print(f"Функция выполнялась {end_time - start_time} секунд.")

Работать это будет. Но посмотрите, сколько здесь дублирования кода! Если нам понадобится изменить формат вывода времени, придется править это в каждой функции. Это неудобно и нарушает один из главных принципов программирования - <b>DRY (Don't Try Yourself)</b>.

Вот для таких задач и были придуманы декораторы.

<b>Декоратор</b> - это функция, которая принимает другую функцию в качестве аргумента, добавляет к ней некоторую новую функциональность и возвращает измененную функцию, не изменяя при этом исходный код самой функции.

Проще всего представить декоратор как <b>"обёртку"</b>:
- У вас есть <b>подарок</b> (ваша исходная функция, например, download_data).
- Вы хотите его украсить, не меняя сам подарок. Вы берете <b>красивую оберточную бумагу и бант</b> (это и есть наш декоратор) и заворачиваете в неё подарок.

Подарок внутри остался прежним, но теперь у него появилось дополнительное свойство - красивая упаковка. Его внешний вид и то, как мы с ним взаимодействуем, изменились.

Точно так же и декоратор: он "оборачивает" вашу функцию в дополнительный код, который может выполняться до или после основной логики вашей функции. При этом сам код download_data остается чистым и нетронутым!

Ключевая мысль, которую нужно запомнить на этом шаге: <b>декоратор позволяет расширить поведение функции, не внося изменений в её исходный код</b>.

Это невероятно мощный инструмент для написания чистого и поддерживаемого кода. В следующих шагах мы разберем, на каких фундаментальных возможностях Python это работает и как выглядит синтаксис декораторов.

### Шаг 2: Ключевые преимущества декораторов

Итак, мы поняли основную идею декоратора как "обёртки". Теперь давайте более детально разберем, какие конкретные выгоды это нам дает и почему декораторы так любят Python-разработчики.

Вспомним нашу проблему из предыдущего шага: нам пришлось скопировать один и тот же код для замера времени в несколько функций. Это привело к дублированию. Декораторы элегантно решают эту и другие подобные проблемы.

Вот их ключевые преимущества:

#### 1. Переиспользование кода (Принцип DRY)

Это самое очевидное преимущество. Логику, которую вы хотите применить ко многим функциям (логирование, замер времени, проверка прав доступа, кэширование), вы выносите в одно-единственное место - в тело декоратора.

- <b>Было</b>: Код для замера времени был скопирован в download_data, process_file и, возможно, в десятки других функций.

- <b>Станет</b>: Мы напишем один декоратор @timer один раз. После этого мы сможем применить его к любой функции, просто добавив одну строчку над её определением.

In [11]:
import time

# Концептуальный пример (сам декоратор мы напишем позже)
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        print(f'Time: {end - start:.4f} s')

        return result
    return wrapper

@timer
def download_data():
    print("Начинаю скачивание...")
    # ... какая-то долгая операция ...
    print("Скачивание завершено.")

@timer
def process_file():
    print("Начинаю обработку файла...")
    # ... какая-то долгая операция ...
    print("Обработка завершена.")

In [12]:
download_data()
print('-' * 30)
process_file()

Начинаю скачивание...
Скачивание завершено.
Time: 0.0002 s
------------------------------
Начинаю обработку файла...
Обработка завершена.
Time: 0.0000 s


Если нам понадобится изменить логику таймера (например, выводить время в миллисекундах), мы изменим код только в одном место - внутри декоратор @timer. Все фунции, отмеченные им, автоматически подхватят новое поведение.

#### 2. Чистота и читаемость кода

Декораторы помогают отделить основную бизнес-логику функции от "служебной" или, как ее еще называют, "сквозной" логики.

Посмотрите на пример выше. С первого взгляда на определение функции def download_data() мы понимаем две вещи:
1. Её основная задача - скачивать данные (это видно из её "чистого" тела).
2. К ней применима дпоолнительная логика замера времени (это видно из названия декоратора @timer).

Код самой функции не загроможден техническими деталями. Он описывает только то, что он делает, а не как он логируется или замеряется. Это делает код гораздо проще для прочтения и понимания.

#### 3. Разделение ответственностей (Separation of Concerns)

Этот пункт тесно связан с предыдущим. Хороший код - это код, где каждый компонент отвечает за что-то одно.

- <b>Ответственность функции download_data</b> - скачивать данные.

- <b>Ответственность декоратора @timer</b> - измерять время выполнения.

Функция ничего не знает о том, что ее время измеряют. Декоратор, в свою очередь, ничто не знает о том, что именно делает функция, которую он оборачивает. Он просто запускает её и замеряет время. Такое разделение делает систему более гибкой и модульной. Вы можете комбинировать разные декораторы с разными функциями, как конструктор Lego.

#### Итог

Декораторы помгают нам:
- <b>Избежать дублирования кода</b>, вынося общую логику в одно место.

- <b>Сделать код чище</b>, так как тело функции содержит только её основную задачу.

- <b>Улучшить читаемость</b>, ведь по названию декоратора сразу понятно, какая дополнительная функциональность применяется.

- <b>Лучше организовать архитектуру кода</b>, разделяя разные логические задачи.

### Шаг 3: Синтаксис декоратора - символ @

Мы уже поняли, что такое декоратор и зачем он нужен. Теперь давайте посмотрим, как он выглядит в коде. В Python для применения декораторв существует специальный, очень удобный и лаконичный синтаксис с использованием символа @.

Этот синтаксис - яркий пример того, что в программировании называют <b>"синтаксическим сахаром"</b>. Это не новая возможность языка, а просто более короткий и приятный способ написать то, что можно было бы написать и более длинным путем.

#### Как это выглядит?

Синтаксис предельно прост: вы пишите имя декоратора, предваряя его символом @, на строке непосредственно перед определением функции (def).

Давайте представим, что у нас уже есть готовый декоратор с имененм my_decorator. Вот как мы применим его к нашей функции:

In [14]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print('Работа декоратора!')
        return func(*args, **kwargs)
    return wrapper

# @имя_декоратора
@my_decorator
def say_hello():
    print("Привет, мир!")

# Теперь, когда мы вызовем функцию...
say_hello()

Работа декоратора!
Привет, мир!


Эта простая запись @my_decorator полностью меняет то, как Python обрабатывает определение функции say_hello. Мы как бы говорим интерпретатору:

"Эй Python! Прежде чем окончательно создать функцию say_hello, возьми её, передай в качестве аргумента в my_decorator, а то, что вернёт my_decorator, и будет теперь новой версией функции say_hello".

#### Что на самом деле происходит "под капотом"?

Запись с символом @ - это всего лишь удобная замена следующей конструкции:

In [16]:
# 1. Сначала мы определяем нашу обычную функцию
def say_hello():
    print("Привет, мир!")

# 2. Затем мы "вручную" применяем декоратор:
#    передаем нашу функцию в него, а результат
#    сохраняем обратно в ту же переменную.
say_hello = my_decorator(say_hello)
say_hello()

Работа декоратора!
Привет, мир!


Эти два блока кода - с @ и без него - <b>абсолютно эквивалентны</b> и делают одно и то же.

Версия с @ предпочтительнее по двум причинам:
1. <b>Она корочие и чище</b>.
2. <b>Она декларативна</b>. Вы сразу видите, что функция "украшена" (декорирована) при её определении, а не где-то дальше в коде.

#### Давайте закрепим на нашем примере с таймером:

Когда мы напишем так:

In [17]:
@timer
def long_calculation():
    # ... какие-то долгие вычисления ...
    return 100

print(long_calculation())

Time: 0.0000 s
100


Python выполним следующие действия:
1. Создаст функцию long_calculation.
2. Сразу же вызовет timer(long_calculation).
3. Результат, который вернет timer, он присвоит переменной long_calculation.

Поэтому при последующем вызове long_calculation() будет выполняться уже не исходная функция, а та новая, которую создал для нас декоратор.

### Задачи

#### Задача 1: Применить декоратор-заголовок

<b>Условие задачи</b>:

В системе уже есть декоратор с именем @header. Он печатает на экран строку "--- HEADER ---" перед тем, как выполнить любую функцию.

Вам дана готовая функция show_message. Ваша задача — <b>применить</b> к ней декоратор @header.

In [19]:
def header(func):
    def wrapper(*args, **kwargs):
        print('--- HEADER ---')
        return func(*args, **kwargs)
    return wrapper


@header
def show_message():
    print('This is the main content.')

#### Задача 2: Применить декоратор двойного вызова

<b>Условие задачи</b>:

В системе есть декоратор @double_it, который заставляет любую декорированную им функцию выполниться ровно два раза.

Примените декоратор @double_it к готовой функции greet.

In [20]:
def double_it(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return wrapper

@double_it
def greet():
    print('Hello')

#### Задача 3: Выбрать и применить правильный декоратор

<b>Условие задачи</b>:

В системе есть два декоратора: @stars (печатает ***) и @dashes (печатает ---).

Вам дана функция print_name. Примените к ней <b>один</b> из этих декораторов, чтобы итоговый вывод был обрамлен тремя звездочками (***).

In [21]:
def stars(func):
    def wrapper(*args, **kwargs):
        print('*' * 3)
        func(*args, **kwargs)
        print('*' * 3)
    return wrapper

def dashes(func):
    def wrapper(*args, **kwargs):
        print('-' * 3)
        func(*args, **kwargs)
        print('-' * 3)
    return wrapper

@stars
def print_name():
    print('Python')

#### Задача 4: Применить "тихий" декоратор

<b>Условие задачи</b>:

В системе есть декоратор @silencer, который полностью подавляет выполнение функции (при ее вызове ничего не выводится на экран).

Примените этот декоратор к функции make_noise, чтобы она ничего не напечатала при вызове.

In [22]:
def silencer(func):
    def wrapper(*args, **kwargs):
        return
    return wrapper

@silencer
def make_noise():
    print('This text should not appear')

#### Задача 5: Применить два декоратора в правильном порядке

<b>Условие задачи</b>:

В системе есть два декоратора: @border (добавляет ========) и @title (добавляет TITLE).
Примените к функции show_report оба декоратора так, чтобы сначала сработал @title, а уже потом, снаружи, @border.

<b>Подсказка</b>: В Python декораторы применяются снизу вверх (от функции наружу).
- Декоратор, который написан <b>ниже</b> (ближе к def), применится первым (внутренняя логика).
- Декоратор, который написан <b>выше</b> (дальше от def), применится последним и станет "внешней" оберткой.

In [23]:
def border(func):
    def wrapper(*args, **kwargs):
        print('=' * 8)
        func(*args, **kwargs)
        print('=' * 8)
    return wrapper

def title(func):
    def wrapper(*args, **kwargs):
        print('TITLE')
        func(*args, **kwargs)
    return wrapper

@border
@title
def show_report():
    print('Report Body')

## Функции как объекты первого класса

### Шаг 1: Демонстрируем, что в Python функция - это такой же объект, как число или строка

В предыдущем уроке мы познакомились с синтаксисом декораторов @. Но чтобы понять, как они работают изнутри, нам нужно сделать небольшой шаг назад и усвоить одну из самых фундаментальных и красивых идей в Python: <b>функции здесь являются объектами первого класса (first-class objects)</b>.

Звучит немного академично, но на самом деле идея очень проста. Это означает, что с функцией можно обращаться так же, как с любым другим, более привычным для нас объектом: например, с числом, строкой или списком.

Давайте посмотрим на знакомые нам объекты. Мы можем легко присвоить их переменным:

In [1]:
# Присваиваем число переменной
x = 10
print(x) # Вывод: 10

# Присваиваем строку переменной
message = "Привет, мир!"
print(message) # Вывод: Привет, мир!

10
Привет, мир!


Это кажется очевидным. Так вот, с функциями в Python можно сделать <b>то же самое</b>.

Давайте определим простую функцию:

In [2]:
def greet():
    print("Это приветствие от функции greet!")

Теперь, вместо того чтобы сразу ее вызвать, давайте присвоим ее новой переменной.

In [3]:
# Присваиваем объект функции greet переменной my_function
my_function = greet

# Проверим, что теперь находится в переменной my_function
print(my_function) # Вывод: <function greet at 0x...> (адрес в памяти может отличаться)

<function greet at 0x000005FCDA124880>


Обратите внимание на важный момент: мы написали greet <b>без круглых скобок</b>.
- greet() - это <b>вызов</b> функции, который выполнит код внутри нее и вернет результат.
- greet - это <b>ссылка</b> на сам объект функции, на ее "тело".

Мы только что сохранили саму функцию в переменную my_function. А раз так, то мы можем вызвать эту функцию, используя новое имя:

In [4]:
my_function() # Вывод: Это приветствие от функции greet!

Это приветствие от функции greet!


Мы получили тот же результат, что и при вызове greet()

#### Давайте докажем, что функция - это объект

Мы можем использовать встроенную функцию type(), чтобы убедиться в этом:

In [5]:
number = 42
text = "hello"

def say_hi():
    pass

print(type(number))   # Вывод: <class 'int'>
print(type(text))     # Вывод: <class 'str'>
print(type(say_hi))   # Вывод: <class 'function'>

<class 'int'>
<class 'str'>
<class 'function'>


Как видите, Python сообщает нам, что тип say_hi - это 'function', точно так же, как он сообщает, что тип number - это 'int'.

Более того, мы можем проверить, что greet и my_function - это просто два разных имени для одного и того же объекта в памяти, используя функцию id():

In [6]:
print(id(greet))           # Вывод: 2284489814464 (пример)
print(id(my_function))     # Вывод: 2284489814464 (пример)
# Идентификаторы совпадают!

6583548528768
6583548528768


#### Главный вывод этого шага:

Функция в Python - это не просто иенованный кусок кода, а полноценный объект, который можно, как и любой другой объект, присвоить переменной.

Осознание этого факта - это ключ к пониманию декораторов. Ведь если мы можем передавать числа и строки в другие функции, то, как мы увидим дальше, мы можем передавать и сами функции.

### Шаг 2: Присваиваем функцию переменной и взываем по новому имени

Итак, в предыдущем шаге мы установили, что функция - это объект. Теперь давайте посмотрим, что это дает нам на практике. Самое первое и простое действие, которое мы можем совершить, - это присвоить этот "объект-функцию" какой-нибудь переменной, точно так же, как мы делаем с числами или строками.

Давайте рассмотрим конкретный пример. Создадим простую функцию, которая переменожает два числа:

In [7]:
def multiply(x, y):
    """Эта функция умножает x на y."""
    return x * y

# Обычный вызов функции
result = multiply(5, 3)
print(f"Обычный результат: {result}")
# Вывод: Обычный результат: 15

Обычный результат: 15


Здесь все стандартно. Теперь сделаем тот самый трюк - <b>присвоим саму функцию</b> новой переменной. Назовем ее, например, operation.

In [8]:
# Важно: мы пишем имя функции БЕЗ круглых скобок!
# Мы присваиваем сам объект, а не результат его вызова.
operation = multiply

Что теперь хранится в переменной operation? Там хранится ссылка на тот же самый объект функции multiply. Мы фактически дали нашей функции второй псевдоним.

Раз operation теперь указывает на нашу функцию, мы можем использовать эту переменную для вызова этой функции.

In [9]:
# Вызываем функцию через новое имя
new_result = operation(5, 3)
print(f"Результат через новую переменную: {new_result}")
# Вывод: Результат через новую переменную: 15

Результат через новую переменную: 15


Как видите, результат абсолютно тот же! Мы успешно вызвали функцию multiply, используя имя operation.

<b>Давайте закрепим на другом примере</b>. Создадим фукнцию, которая преобразует текст в верхний регистр и добавляет восклицательные знаки.

In [10]:
def scream(text):
    return text.upper() + "!!!"

# Присваиваем функцию переменной 'shout'
shout = scream

# Используем оба имени для вызова
original_output = scream("привет")
new_output = shout("hello")

print(original_output)  # Вывод: ПРИВЕТ!!!
print(new_output)       # Вывод: HELLO!!!

ПРИВЕТ!!!
HELLO!!!


Это работае для любой функции, независимо от того, какие аргументы она принимает и что возвращает.

#### Ключевая мысль, которую нужно вынести из этого шага:

Самое важное - это понимать разницу между my_function и my_function():
- my_function - это <b>ссылка на объект функции</b>. Вы можете присвоить ее переменной, положить в список, передать в другую функцию.

- my_function() - это <b>вызов функции</b>. Python немедленно выполнит код внутри нее и вернет результат.

Это очень просто, но невероятно важно. Когда вы можете "упаковать" функцию в переменную, открывается следующая потрясающая возможность: передавать эти переменные (а значит, и сами функции) в другие функции.

### Шаг 3: Передаем функции в качестве аргументов (Функции высшего порядка)

Мы сделали два важных шага: поняли, что функция - это объект и научились присваивать ее переменным. Теперь мы готовы объединить эти знания и сделать следующий логический шаг.

Если функция - это такой же объект, как число или строка, а мы можем передавать числа и строки в другие функции в качестве аргументов, то... можем ли мы передать <b>функцию в качестве аргумента в другую функцию</b>?

Ответ: <b>Да!</b> И это одна из самых мощных возможностей Python.

<b>Функции высшего порядка (Higher-Order Functions)</b> - это функции, которые могут принимать другие функции в качестве аргументов и/или возвращать функции в качестве результата.

Давайте сразу разберем это на простом и наглядном примере.

Предположим, у нас есть две простые математические функции:

In [11]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

Теперь создадим нашу первую функцию высшего порядка. Назовем ее calculate. Она будет принимать три аргумента:
1. action - здесь мы ожидаем получить <b>объект функции</b> (например, add или subtract).
2. x - первое число.
3. y - второе число.

In [12]:
def calculate(action, x, y):
    """
    Эта функция принимает другую функцию (action)
    и применяет ее к аргументам x и y.
    """
    # Здесь мы ВЫЗЫВАЕМ функцию, которую получили в аргументе 'action'
    result = action(x, y)
    print(f"Результат выполнения операции: {result}")

# --- А теперь используем нашу функцию высшего порядка! ---

# 1. Передаем функцию 'add' в качестве первого аргумента
calculate(add, 10, 5)
# Вывод: Результат выполнения операции: 15

# 2. Передаем функцию 'subtract' в качестве первого аргумента
calculate(subtract, 10, 5)
# Вывод: Результат выполнения операции: 5

Результат выполнения операции: 15
Результат выполнения операции: 5


#### Что здесь произошло?

1. В первом вызове calculate(add, 10, 5) переменная action внутри функции calculate стала ссылаться на объект функции add. Поэтому строка result = action(x, y) превратилась в result = add(10, 5).

2. Во втором вызове calculate(subtract, 10, 5) переменная action уже ссылалась на subtract. И та же самая строка result = action(x, y) выполнилась как result = subtract(10, 5).

Наша функция calculate получилась очень гибкой! Она не знает, какую именно операцию выполняет, она просто делегирует эту работу той функции, которую ей передали.

#### Как это связано с декораторами?

Эта концепция - самое сердце механизма работы декораторов. Вспомните синтаксис:

In [14]:
# @my_decorator
# def my_function():
#     pass

Это всего лишь красивый способ сказать:

In [15]:
# my_function = my_decorator(my_function)

Как видите, my_decorator - это и есть <b>функция высшего порядка</b>, потому что она принимает другую функцию (my_function) в качестве аргумента!

#### Итог

Способность передавать функции в качестве аргументов - это не просто интересный трюк, а фундаментальный принцип, на котором строятся декораторы. Мы создаем одну функцию ("декоратор"), которая будет принимать и "оборачивать" другие.

Мы почти у цели! Нам осталось разобраться с последним кусочком пазла - замыканиями, и после этого мы будем полностью готовы к написанию нашего первого полноценного декоратора.

### Шаг 4: Изучаем концепцию возвращения функции из другой функции

Мы научились передавать функции внуть других функций. Теперь давайте посмотрим на обратный процесс: как одна функция может создавать и возвращать другую функцию.

Если вы увоили предыдущие шаги, эта идея покажется вам вполне логичной. Раз функция - это объект, то ее, как и любой другой объект (число, строку, список), можно вернуть с помощью оператора return.

Представьте себе "фабрику", которая производит не детали, а готовые к работе функции. Вы делаете заказ на "фабрику" (вызываете внешнюю функцию с определенными параметрами), а она возвращает вам новую, настроенную под ваш заказ функцию.

Давайте напишем такую "фабрику приветствий".

In [17]:
def create_greeter(language):
    """
    Эта функция-фабрика создает и возвращает
    одну из вложенных функций в зависимости от языка.
    """
    
    # Определяем вложенную функцию (она существует только здесь)
    def greet_in_english(name):
        return f"Hello, {name}!"
        
    # Определяем другую вложенную функцию
    def greet_in_spanish(name):
        return f"Hola, {name}!"
        
    # Фабрика решает, какой "продукт" вернуть
    if language == "en":
        return greet_in_english  # Возвращаем объект функции, БЕЗ ()
    elif language == "es":
        return greet_in_spanish  # Возвращаем объект функции, БЕЗ ()

# --- Используем нашу фабрику ---

# 1. Заказываем "английскую" версию приветствия
english_greeter = create_greeter("en")
print(english_greeter) 

# 2. Заказываем "испанскую" версию
spanish_greeter = create_greeter("es")
print(spanish_greeter)

<function create_greeter.<locals>.greet_in_english at 0x000005FCDA125820>
<function create_greeter.<locals>.greet_in_spanish at 0x000005FCDA124BA0>


Посмотрим на вывод: переменные english_greeter и spanish_greeter теперь содержат не строки, а самые настоящие <b>объекты функций</b>. Обратите внимание, что Python даже указывает нам, что эти функции (greet_in_english и greet_in_spanish) были созданы внутри (<locals>) функции create_greeter.

А раз у нас есть объекты функций, мы можем их вызвать!

In [18]:
# Вызываем функцию, которую нам вернула фабрика
print(english_greeter("Alice"))  # Вывод: Hello, Alice!
print(spanish_greeter("Bob"))    # Вывод: Hola, Bob!

Hello, Alice!
Hola, Bob!


#### Как это всё связано с декоратором?

Теперь у нас есть все составные части для понимания полной механики декоратора. Давайте соберем их вместе.

<b>Декоратор - это функция, которая</b>:
1. <b>Принимает</b> другую функцию в качестве аргумента.

2. <b>Создает</b> внутри себя новую, "оберточную" функцию (которая будет содержать дополнительную логику).

3. <b>Возвращает</b> эту новую "оберточную" функцию.

Эта возврщенная "обертка" и есть то, что заменяет нашу исходную функцию, когда мы используем синтаксис @.

### Задачи

#### Задача 1: Присвоить функцию переменной

<b>Условие задачи</b>:

Вам дана готовая функция say_loudly(). (она уже написана в коде, просто не видна).

Создайте новую переменную с именем say_quietly и присвойте ей <b>саму функцию</b> say_loudly.

Подсказка: при присваивании функции переменной не используйте круглые скобки ().

In [19]:
# say_quietly = say_loudly

#### Задача 2: Написать функцию-исполнитель

<b>Условие задачи</b>:

Напишите функцию высшего порядка с именем caller(any_func). Эта функция должна принимать одну любую функцию any_func в качестве аргумента и просто вызывать ее.

In [20]:
def caller(any_func):
    any_func()

#### Задача 3: Написать функцию тройного вызова

<b>Условие задачи</b>:

Напишите функцию высшего порядка с именем do_three_times(func_to_run). Эта функция должна принимать одну любую функцию в качестве аргумента и вызывать ее ровно три раза.

In [21]:
def do_three_times(func_to_run):
    func_to_run()
    func_to_run()
    func_to_run()

#### Задача 4: Выборочный исполнитель

<b>Условие задачи</b>:

Напишите функцию run_selected(functions_list, name).
Она принимает два аргумента:
1. functions_list — список функций.
2. name — строка.

Функция должна перебрать список functions_list и вызвать только ту функцию, у которой имя (\_\_name__) совпадает со строкой name.

In [23]:
def run_selected(functions_list, name):
    for func in functions_list:
        if func.__name__ == name:
            func()
            break

#### Задача 5: Создать словарь операций

<b>Условие задачи</b>:

Вам даны две готовые функции: get_greeting() и get_farewell().(они уже написаны в коде, просто не видна)
Создайте словарь (dict) с именем MESSAGES. В этом словаре должно быть два ключа:
1. Строка "welcome", значением для которой должна быть функция get_greeting.
2. Строка "goodbye", значением для которой должна быть функция get_farewell.

In [24]:
def get_greeting(): pass
def get_farewell(): pass

MESSAGES = {
    'welcome': get_greeting,
    'goodbye': get_farewell
}

## Вложенные функции и замыкания

### Шаг 1: Разбираем, как и зачем определять функции внутри других функций (вложенные функции)

Здраствуйте! В предыдущем уроке, когда мы создавали нашу "фабрику приветствий" create_greeter, вы, возможно, заметил кое-что интересное: мы определяли одни функции (greet_in_english, greet_in_spanish) прямо внутри другой функции.

In [1]:
def create_greeter(language):
    # 👇 Вот эти функции мы и обсуждаем
    def greet_in_english(name):
        return f"Hello, {name}!"
        
    def greet_in_spanish(name):
        return f"Hola, {name}!"
    # ...

Такая техника имеет свое название. Функция, определенная внутри другой функции, называется <b>вложенный функцией</b> (nested function).

#### Как это работает?

Правила здесь очень простые. Вложенная функция создается каждый раз, когда вызывается внешняя, "родительская" функция, и она существует только внутри этой родительской функции. Это означает, что у нее есть своя <b>область видимости (scope)</b>.

Давайте посмотрим на простой пример:

In [2]:
def parent_function():
    print("Я - родительская функция.")
    
    # Определяем вложенную функцию
    def child_function():
        print("А я - вложенная, дочерняя функция.")
        
    # Вызываем вложенную функцию из родительской
    child_function()

# Вызываем родительскую функцию. Это сработает.
parent_function()

Я - родительская функция.
А я - вложенная, дочерняя функция.


Все логично. Но что произойдет, если мы попытаемся вызвать child_function() напрямую, извне?

In [3]:

try:
    child_function()
except NameError as e:
    print(f"Ошибка! {e}")

Ошибка! name 'child_function' is not defined


child_funtion "живет" только внутри parent_function. Вне этого контекста ее просто не существует.

#### Зачем это нужно?

Может показаться, что это просто способ усложнить код, но на самом деле у вложенных функций есть несколько важных применений.

##### 1. Инкапсуляция и сокрытие логики.

Иногда ввам нужна небольшая вспомогательная функция, которая выполняет какую-то утилитарную задачу, но эта задача нужна только внутри одной другой, более крупной функции. Вынося ее наружу, вы "загрязняете" общее пространство имен. Определяя ее внутри, вы четко показываете: "эта функция - деталь реализации, она не предназначена для использования где-либо еще".

Например:

In [4]:
def process_data(data_list):
    
    def format_line(line):
        # Сложное форматирование, нужное только здесь
        return line.strip().upper()

    processed_data = [format_line(line) for line in data_list]
    return processed_data

Здесь format_line - это внутренняя деталь process_data, и нет никакого смысла делать ее доступной глобально.

##### 2. Подготовка к замыканиям и декораторам.

И это - <b>главная причина</b>, по которой мы изучаем вложенные функции.

Вложенная функция имеет доступ не только к своим собственным аргументам и переменным, но и к переменным из "родительской" функции. Эта способность "помнить" окружение, в котором она была создана, и является основной для механизма <b>замыканий</b>, который мы разберем в следующем шаге.

А замыкания, в свою очередь, - это сердце любого классического декоратора.

### Шаг 2: Вводим понятие замыкания (closure). Объясняем, как внутренняя функция "помнит" свое окружение

Мы увидели, что вложенные функции имеют доступ к переменным родительской функции. Но настоящее "волшебство" начинается тогда, когда родительская функция <b>возвращает</b> вложенную функцию. Оказывается, эта возвращенная функция <b>"помнит"</b> те переменные из родительской функции, которые ей нужны, даже после того, как родительская функция уже завершила свою работу и все ее локальные переменные должны были бы исчезнуть.

Это явление и называется <b>замыканием (closure)</b>.

<b>Замыкание</b> - это объект функции, который "помнит" и имеет доступ к переменным из этой области видимости, в которой он был создан, даже если эта область видимости больше не существует.

Лучший способ понять это - на примере. Давайте создадим "фабрику", которая производит функции-умножители.

In [5]:
def multiplier_factory(factor):
    """Эта фабрика создает функцию, которая будет умножать на 'factor'."""
    
    print(f"--- Фабрика была запущена с фактором {factor}. ---")
    
    # Вложенная функция. Она использует 'factor' из своего окружения.
    def multiplier(number):
        return number * factor
        
    # Фабрика возвращает вложенную функцию
    return multiplier

Теперь давайте воспользуемся этой фабрикой.

In [6]:
# 1. Создаем функцию, которая всегда будет умножать на 5
times_5 = multiplier_factory(5)

# 2. Создаем другую функцию, которая всегда будет умножать на 10
times_10 = multiplier_factory(10)

print("\n--- Фабрики завершили свою работу. Теперь у нас есть две новые функции. ---\n")

--- Фабрика была запущена с фактором 5. ---
--- Фабрика была запущена с фактором 10. ---

--- Фабрики завершили свою работу. Теперь у нас есть две новые функции. ---



Обратите внимание: функция multiplier_factory уже <b>выполнилась и завершилась</b>. Дважды. Локальная переменная factor в каждом из этих вызовов должна была быть уничтожена после заверешния работы функции.

Но давайте посмотрим, что произойдет, когда мы вызовем созданные нами функции:

In [7]:
# Вызываем первую созданную функцию
result1 = times_5(4)
print(f"Результат times_5(4): {result1}")

# Вызываем вторую созданную функцию
result2 = times_10(4)
print(f"Результат times_10(4): {result2}")

Результат times_5(4): 20
Результат times_10(4): 40


#### Как это работает? В чем магия?

Функция times_5 каким-то образом <b>помнит</b>, что ее factor был равен 5. А times_10 <b>помнит</b>, что ее factor был равен 10.

Это и есть замыкание в действии. Когда multiplier_factory возвращала внутреннюю функцию multiplier, Python увидел, что multiplier использет переменную factor из своего окружения. Поэтому он "замкнул" (closed over) эту переменную, сохранив ее значение вместе с кодом функции.

Можно представить, что возвращенная функция multiplier несет с собой маленький "рюкзачок", а котором лежат все переменные из родительской функции, которые ей могут понадобиться. У times_5 в рюкзачке лежит factor = 5, а у times_10 = 10.

#### Три условия для создания замыкания:

1. У нас должна быть вложенная функция.

2. Вложенная функция должна ссылаться на переменную из родительской (внешней) функции.

3. Родительская функция должна возвращать вложенную функцию.

#### Как это связано с декораторами?

Это и есть недостающий элемент! Функция-обертка, которую мы создаем внутри декоратора, <b>является замыканием</b>. Она "запоминает" функцию, которую нужно декорировать (func), даже после того, как сам декоратор отработал. Именно поэтому обертка может вызвать исходную func когда угодно позже.

#### Итог

Замыкание - это функция, у которой есть "память". Она запоминает свое лексическое окружение (переменные из родительской функции) и может использовать их в будущем. Это фундаментальный механизм, который делает возможной работу декораторов.

### Шаг 3: Иллюстрируем замыкание на простом и наглядном примере

Давайте закрепим наше понимание замыканий на классическом и очень показательном примере - <b>создании счетчика</b>.

<b>Задача</b>: Мы хотим создать функцию-счетчик. Каждый раз, когда мы ее вызываем, она должна возвращать число, на единицу большее, чем в предыдущий раз (1, 2, 3, ...). При этом мы хотим иметь возможность создавать несколько таких счетчиков, и каждый из них должен вести свой собственный, независимый счет.

Использовать для этого классы или глобальные переменные не будем - это было бы слишком просто и не так интересно! Мы решим эту задачу с помощью замыкания.

Вот как будет выглядеть наша "фабрика счетчиков":

In [8]:
def counter_factory():
    """Фабрика, которая создает и возвращает функцию-счетчик."""
    
    # Эта переменная 'count' создается в момент запуска фабрики.
    # Она будет "замкнута" внутри вложенной функции.
    count = 0
    
    # Вложенная функция, которая и будет нашим счетчиком.
    def increment():
        # 'nonlocal' говорит, что мы хотим изменить переменную 'count'
        # не из этой функции, а из родительской.
        nonlocal count 
        count += 1
        return count
        
    # Фабрика возвращает готовую к работе функцию-счетчик.
    return increment

# --- Давайте проверим, как это работает! ---

# 1. Создаем наш первый счетчик
counter_A = counter_factory()

print("Вызываем первый счетчик (A):")
print(counter_A())  # Вывод: 1
print(counter_A())  # Вывод: 2
print(counter_A())  # Вывод: 3

# 2. Теперь создадим ВТОРОЙ, совершенно независимый счетчик
counter_B = counter_factory()

print("\nВызываем второй счетчик (B):")
print(counter_B())  # Вывод: 1
print(counter_B())  # Вывод: 2

print("\nСнова вызываем первый счетчик (A), чтобы убедиться, что он не сбился:")
print(counter_A())  # Вывод: 4

Вызываем первый счетчик (A):
1
2
3

Вызываем второй счетчик (B):
1
2

Снова вызываем первый счетчик (A), чтобы убедиться, что он не сбился:
4


#### Анализ примера

Этот пример идеально иллюстрирует "память" замыкания:

1. <b>Создание Counter_A</b>: Когда мы вызвали counter_factory(), была создана локальная переменная count со значение 0. Затем была создана и возвращена функция increment. Эта функция "замкнула" в себе переменную count. Теперь counter_A - это функция increment, которая несет с собой свою собственную, приватную "ячейку памяти", где хранится count = 0.

2. <b>Вызовы counter_A</b>: Каждый вызов counter_A() обращается к этой приватной "ячейке памяти", увеличивает ее значение на 1 и возвращает результат.

3. <b>Создание counter_B</b>: Когда мы снова вызвали counter_factory(), <b>весь процесс повторился с нуля</b>. Была создана <b>новая</b> локальная переменная count со значением 0 и <b>новая</b> функция increment, которая замкнула в себе уже эту, вторую переменную.

В итоге counter_A и counter_B - это две совершенно разные функции-замыкания. У каждой из них свой собственный, изолированный "рюкзачок" с переменными. Именно поэтому счет counter_A никак не влияет на счет counter_B.

#### Что такое nonlocal?

Вы заметили новое ключевое слово <b>nonlocal</b>. Оно очень важно. Если бы мы просто написали count += 1 внутри increment(), Python решил бы, что мы пытаемся создать новую локальную переменную count внутри increment, и выдал бы ошибку.

Ключевое слово <b>nonlocal count</b> явно указывает: "Я хочу работать не с локальной переменной, а с переменной count из ближайшей родительской функции".

#### Итог

Замыкание - это не просто функция, которая читает переменные из своего окружения, но и которая может изменять их. Это позволяет нам создавать функции, обладающие <b>состоянием</b> (state) - то есть способностью помнить информацию между вызовами. Эта концепция является основной для многих продвинутых декораторов.

### Задачи

#### Задача 1: Фабрика приветствий

<b>Условие задачи</b>:

Напишите функцию-фабрику greeter_factory(greeting).
Эта функция должна принимать один аргумент greeting (строку). Внутри себя она должна определять и возвращать <b>новую вложенную функцию</b>.

Возвращаемая вложенная функция должна принимать один аргумент name (строку) и возвращать строку в формате f"{greeting}, {name}!".

In [15]:
def greeter_factory(greeting):
    def greet_to(name):
        return f'{greeting}, {name}!'
    return greet_to

#### Задача 2: Фабрика умножителей

<b>Условие задачи</b>:

Напишите функцию-фабрику multiplier_factory(factor).

Эта функция должна принимать один числовой аргумент factor. Она должна возвращать новую функцию, которая, в свою очередь, принимает один числовой аргумент number и возвращает результат умножения number на factor.

In [ ]:
def multiplier_factory(factor):
    def multiply(number):
        return factor * number
    return multiply

#### Задача 3: Простой счетчик

<b>Условие задачи</b>:

Напишите функцию-фабрику counter_factory().
Эта функция не принимает аргументов. Она должна возвращать новую функцию-счетчик.

Каждый вызов возвращенной функции должен возвращать число на 1 больше, чем в предыдущий раз. Первый вызов должен вернуть 1, второй — 2, и так далее.

Подсказка: вам понадобится переменная для хранения счета в замыкании и ключевое слово nonlocal для ее изменения.

In [17]:
def counter_factory():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

#### Задача 4: Хранилище знаний

<b>Условие задачи</b>:

Напишите функцию-фабрику storage_factory(initial_value).
Она принимает начальное значение initial_value. Фабрика должна возвращать <b>две</b> функции в виде кортежа (getter, setter):
1. getter(): при вызове возвращает текущее хранящееся значение.
2. setter(new_value): принимает новое значение и обновляет хранящееся.

In [18]:
def storage_factory(initial_value):
    value = initial_value

    def getter():
        return value

    def setter(new_value):
        nonlocal value
        value = new_value

    return getter, setter

#### Задача 5: Фабрика с условием

<b>Условие задачи</b>:

Напишите функцию-фабрику conditional_function_factory(condition).
Она принимает булево значение condition.
- Если condition равно True, фабрика должна вернуть функцию, которая при вызове печатает "Action A".
- Если condition равно False, фабрика должна вернуть функцию, которая при вызове печатает "Action B".

In [21]:
def conditional_function_factory(condition):
    if condition:
        return lambda: print('Action A')
    return lambda: print('Action B')

## Создание первого простого декоратора

### Шаг 1: Создаем декоратор "вручную", без синтаксиса @, чтобы понять механику его работы

Мы изучали функции как объекты, вложенные функции и замыкания. Теперь мы сложим эти три концепции вместе, чтобы построить наш первый настоящий декоратор.

<b>Наша задача</b>: создать декортаор, который будет выводить сообщение в консоль до и после вызова любой функции.

#### 1. Наша исходная функция (которую будем "укарашать")

Давайте возьмем самую простую функцию, какую только можно представить:

In [23]:
def say_hello():
    print("Привет, мир!")

say_hello()

Привет, мир!


#### 2. Создание функции-декоратора

Теперь напишем наш декоратор. Вспомним его структуру, основанную на замыканиях:
- Это внешняя функция, которая <b>принимает</b> другую функцию в качестве аргумента.
- Внутри нее определяется <b>вложенная</b> функция-обертка.
- Обертка <b>добавляет</b> новую логику и <b>вызывает</b> исходную функцию.
- Внешняя функция <b>возвращает</b> обертку.

In [24]:
def simple_decorator(func_to_decorate):
    """Это наш первый, самый простой декоратор."""
    
    # 1. Определяем вложенную функцию-"обертку"
    def wrapper():
        # 2. Добавляем новый функционал ПЕРЕД вызовом исходной функции
        print("--- Что-то происходит ПЕРЕД вызовом функции. ---")
        
        # 3. Вызываем исходную функцию, которую мы "замкнули"
        func_to_decorate()
        
        # 4. Добавляем новый функционал ПОСЛЕ вызова исходной функции
        print("--- Что-то происходит ПОСЛЕ вызова функции. ---")
        
    # 5. Декоратор возвращает саму функцию-обертку (НЕ вызывая её!)
    return wrapper

#### 3. Применение декоратора "вручную"

Теперь самое интересное. Как нам применить этот декоратор к нашей функции say_hello? Мы используем тот самый синтаксис, который прячется за символом @:

In [25]:
# 'say_hello' пока еще наша оригинальная, простая функция
print("Оригинальная функция:", say_hello)

# Теперь мы "декорируем" её:
# 1. Передаем 'say_hello' в наш декоратор.
# 2. 'simple_decorator' возвращает нам функцию 'wrapper'.
# 3. Мы сохраняем эту новую функцию в переменную с тем же именем 'say_hello'.
say_hello = simple_decorator(say_hello)

# 'say_hello' теперь ссылается на новую, "обернутую" функцию
print("Функция после декорирования:", say_hello)

Оригинальная функция: <function say_hello at 0x00000253723B2C60>
Функция после декорирования: <function simple_decorator.<locals>.wrapper at 0x00000253723B8B60>


Переменная say_hello была <b>перезаписана</b>. Раньше она указывала на нашу оригинальную функцию, а теперь она указывает на функцию wrapper, которая была создана и возвращена декоратором. При этом wrapper "помнит" (благодаря замыканию) нашу исходную функцию в своей переменной func_to_decorate.

#### 4. Вызов декорированной функции

Давайте посмотрим, что произойдет, если мы теперь вызовем say_hello():

In [26]:
print("\nВызываем декорированную функцию:")
say_hello()


Вызываем декорированную функцию:
--- Что-то происходит ПЕРЕД вызовом функции. ---
Привет, мир!
--- Что-то происходит ПОСЛЕ вызова функции. ---


Это сработало! Мы не меняли исходный код say_hello, но ее поведение расширилось.

### Шаг 2: Применение синтаксического сахара @ для упрощения кода

В предыдущем шаге мы успешно создали и применили наш первый декоратор. Мы сделали это "вручную", чтобы увидеть механизм в действии. Наш финальный код выглядел так:

In [27]:
# 1. Определяем декоратор
def simple_decorator(func_to_decorate):
    def wrapper():
        print("--- Что-то происходит ПЕРЕД вызовом функции. ---")
        func_to_decorate()
        print("--- Что-то происходит ПОСЛЕ вызова функции. ---")
    return wrapper

# 2. Определяем исходную функцию
def say_hello():
    print("Привет, мир!")

# 3. Применяем декоратор вручную
say_hello = simple_decorator(say_hello)

# 4. Вызываем результат
say_hello()

--- Что-то происходит ПЕРЕД вызовом функции. ---
Привет, мир!
--- Что-то происходит ПОСЛЕ вызова функции. ---


Этот код абсолютно корректен, и он наглядно показывает, что происходит. Но согласитесь, он немного громоздкий. Строка say_hello = simple_decorator(say_hello) находится отдельно от определения самой функции, и ее можно случайно пропустить при чтении кода.

Специально для таких случаев создатели Python добавили в язык "синтаксический сахар" - элегантный и короткий способ сделать то же самое. Это <b>символ @</b>.

Давайте перепишем наш пример с использованием этого синтаксиса.

In [28]:
# 1. Определяем декоратор (он остается точно таким же)
def simple_decorator(func_to_decorate):
    def wrapper():
        print("--- Что-то происходит ПЕРЕД вызовом функции. ---")
        func_to_decorate()
        print("--- Что-то происходит ПОСЛЕ вызова функции. ---")
    return wrapper

# 2. Применяем декоратор к функции в момент её определения
@simple_decorator
def say_hello():
    print("Привет, мир!")

# 3. Просто вызываем функцию!
say_hello()

--- Что-то происходит ПЕРЕД вызовом функции. ---
Привет, мир!
--- Что-то происходит ПОСЛЕ вызова функции. ---


Эти два примера - с ручным присваиванием и с символом @ - <b>абсолютно эквивалентны</b>.

Запись @simple_decorator над определением функции say_hello - это просто команда для Python, которая означает:

"После того как создашь функцию say_hello, не останавливайся. Сразу же передай ее в simple_decorator и замени исходную say_hello тем, что вернется в результате."

#### Почему этот способ лучше?

- <b>Читаемость</b>: Вы сразу видите, что функция say_hello не простая, а "украшенная" дополнительной логикой. Намерение очевидно с первого взгляда.

- <b>Краткость</b>: Меньше кода - меньше шансов на ошибку.

- <b>Декларативность</b>: Мы объявляем (декларируем) наше намерение декорировать функцию в момент её создания, а не делаем это где-то позже в коде.

### Шаг 3: Структура "обертки": логика до, во время и после вызова

Давайте еще раз внимательн посмотрим на сердце нашего декоратора - на вложенную функцию wrapper. Именно в ней и происходит вся магия. Ее структура - это шаблон, который вы будете использовать в 80% случаев при написании декораторов.

In [29]:
def simple_decorator(func_to_decorate):
    
    # Вот она, наша "обертка"
    def wrapper():
        # ===================================================
        # ЧАСТЬ 1: Код, выполняемый ПЕРЕД вызовом оригинала
        # ===================================================
        print("--- Что-то происходит ПЕРЕД вызовом функции. ---")
        
        # ===================================================
        # ЧАСТЬ 2: Непосредственный вызов исходной функции
        # ===================================================
        func_to_decorate() # <<<<<<< Ключевой момент!
        
        # ===================================================
        # ЧАСТЬ 3: Код, выполняемый ПОСЛЕ вызова оригинала
        # ===================================================
        print("--- Что-то происходит ПОСЛЕ вызова функции. ---")
        
    return wrapper

Давайте проанализируем каждую из этих трех частей.

#### Часть 1: Логика "ДО"

Все, что вы пишете внутри wrapper до строки func_to_decorate(), будет выполнено <b>перед</b> тем как запустится основная логика декорируемой функции.

Что здесь можно делать?
- <b>Логирование</b>: Записать в лог, что функция была вызвана, с какими параметрами.
- <b>Проверка условий</b>: Проверить права доступа пользователя или корректность аргументов (мы научилимся работать с аргументами позже).
- <b>Подготока ресурсов</b>: Открыть файл или установить соединение с базой данных.
- <b>Запуск таймера</b>: Засечь время начала выполнения.

В нашем примере мы просто выводим сообщение "--- Что-то происходит ПЕРЕД вызовом функции ---".

#### Часть 2: Вызов исходной функции

Строка func_to_decorate() - это центральная и самая важная часть обертки. Здесь мы "пускаем в ход" ту самую оригинальную фукцию, которую "обернули".

Именно в этот момент выполняется код из say_hello() и на экране появляется "Привет, мир!".

Если вы забудете или закомментируете эту строку, ваша исходная функция <b>никогда не будет вызвана</b>! Декоратор просто "проглотит" ее.

#### Часть 3:  Логика "ПОСЛЕ"

Все, что вы пишете после строки func_to_decorate(), будет выполнено, когда исходная функция уже <b>завершила</b> свою работу.

Что здесь можно делать?
- <b>Обарботка результата</b>: Если функция что-то возвращает, можно этот результат перехватить, изменить, залогировать.
- <b>Очистка ресурсов</b>: Закрыть файл или соединение с базой данных, которые были открыты в Части 1.
- <b>Остановка таймера</b>: Засечь время окончания и посчитать длительность выполнения.
- <b>Отправка уведомлений</b>: Сообщить, что задача успешно выполнена.

В нашем примере мы выводим сообщение "--- Что-то происходит ПОСЛЕ вызова функции. ---".

#### Итог

Функция-обертка дает нам полный контроль над процессом вызова другой функции. Она предоставляет нам три четко определенных места для встраивания нашего кода: <b>до</b>, <b>во время</b> (сам вызов) и <b>после</b>. Понимание этой трехчастной структур - ключ к написанию любых, даже самых сложных декораторов.

### Задачи

#### Задача 1: Декоратор "Старт-Финиш"

<b>Условие задачи</b>:

Напишите декоратор start_finish_decorator.

Декоратор должен "оборачивать" вызов функции. При вызове декорированной функции он должен:
1. Напечатать на экран строку "Start".
2. Вызвать саму функцию.
3. Напечатать строку "Finish".

In [30]:
def start_finish_decorator(func):
    def wrapper(*args, **kwargs):
        print('Start')
        func(*args, **kwargs)
        print('Finish')
    return wrapper

#### Задача 2: Декоратор-разделитель

<b>Условие задачи</b>:

Напишите декоратор separator_decorator. Он должен печатать строку из десяти звездочек (**********) до и после вызова декорируемой функции.

In [32]:
def separator_decorator(func):
    line = '*' * 10

    def wrapper(*args, **kwargs):
        print(line)
        func(*args, **kwargs)
        print(line)

    return wrapper

#### Задача 3: Декоратор двойного вызова

<b>Условие задачи</b>:

Напишите декоратор double_call_decorator. Этот декоратор должен вызывать декорируемую им функцию ровно два раза.

In [33]:
def double_call_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return wrapper

#### Задача 4: Декоратор, который ничего не делает

<b>Условие задачи</b>:

Напишите декоратор dummy_decorator. Этот декоратор должен просто вызывать декорируемую функцию и ничего больше. Никакой дополнительной логики "до" или "после" быть не должно.

Это задание проверяет понимание базовой структуры декоратора.

In [34]:
def dummy_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
    return wrapper

#### Задача 5: Декоратор "Тишина"

<b>Условие задачи</b>:

Напишите декоратор silencer_decorator. Этот декоратор должен <b>полностью подавлять</b> выполнение декорируемой функции. То есть при вызове декорированной функции ничего не должно происходить.

Подсказка: функция-обертка просто не должна вызывать исходную функцию.

In [36]:
def silencer_decorator(func):
    def wrapper(*args, **kwargs):
        pass
    return wrapper